In [1]:
import numpy as np
import pandas as pd
from numba import njit
from jupyter_dash import JupyterDash
from dash import html, dcc, Input, Output, State, dash_table
import plotly.graph_objects as go
import plotly.express as px

In [2]:
# Global config
SEED = 42
np.random.seed(SEED)

TRAIN_PATH = "train_ratings.csv"
VAL_PATH = "val_ratings.csv"
TEST_PATH = "test_ratings.csv"

In [3]:
# Load shared split csv files
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Train shape: (800204, 4)
Val shape: (100002, 4)
Test shape: (100003, 4)


,user_id,movie_id,rating,timestamp
0,3650,1791,3,966459999
1,2857,2365,3,972507382
2,3789,1034,5,966021326
3,4682,648,3,965531092
4,5268,2791,5,961171110


In [4]:
# Basic checks

total_n = len(train_df) + len(val_df) + len(test_df)

print("Train proportion:", len(train_df) / total_n)
print("Val proportion:", len(val_df) / total_n)
print("Test proportion:", len(test_df) / total_n)

print("Train rating min/max:", train_df["rating"].min(), train_df["rating"].max())
print("Val rating min/max:", val_df["rating"].min(), val_df["rating"].max())
print("Test rating min/max:", test_df["rating"].min(), test_df["rating"].max())

required_cols = {"user_id", "movie_id", "rating"}
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{name} file missing columns: {missing}")

Train proportion: 0.8000367923104071
Val proportion: 0.0999811039492746
Test proportion: 0.09998210374031827
Train rating min/max: 1 5
Val rating min/max: 1 5
Test rating min/max: 1 5


In [5]:
# Build user/movie mappings
all_user_ids = pd.Index(
    pd.concat([train_df["user_id"], val_df["user_id"], test_df["user_id"]]).unique()
)
all_movie_ids = pd.Index(
    pd.concat([train_df["movie_id"], val_df["movie_id"], test_df["movie_id"]]).unique()
)

user2idx = {uid: i for i, uid in enumerate(all_user_ids)}
movie2idx = {mid: i for i, mid in enumerate(all_movie_ids)}

idx2user = {i: uid for uid, i in user2idx.items()}
idx2movie = {i: mid for mid, i in movie2idx.items()}

n_users = len(user2idx)
n_items = len(movie2idx)

print("Number of users:", n_users)
print("Number of movies:", n_items)

Number of users: 6040
Number of movies: 3706


In [6]:
# Map raw ids to internal indices
def map_df(df):
    out = df.copy()
    out["u_idx"] = out["user_id"].map(user2idx)
    out["i_idx"] = out["movie_id"].map(movie2idx)
    return out

train_df = map_df(train_df)
val_df = map_df(val_df)
test_df = map_df(test_df)

display(train_df.head())

,user_id,movie_id,rating,timestamp,u_idx,i_idx
0,3650,1791,3,966459999,0,0
1,2857,2365,3,972507382,1,1
2,3789,1034,5,966021326,2,2
3,4682,648,3,965531092,3,3
4,5268,2791,5,961171110,4,4


In [7]:
# Convert to numpy arrays
def df_to_arrays(df):
    return (
        df["u_idx"].values.astype(np.int64),
        df["i_idx"].values.astype(np.int64),
        df["rating"].values.astype(np.float64)
    )

u_train, i_train, y_train = df_to_arrays(train_df)
u_val, i_val, y_val = df_to_arrays(val_df)
u_test, i_test, y_test = df_to_arrays(test_df)

print("Sample train arrays:")
print("u_train[:5] =", u_train[:5])
print("i_train[:5] =", i_train[:5])
print("y_train[:5] =", y_train[:5])

Sample train arrays:
u_train[:5] = [0 1 2 3 4]
i_train[:5] = [0 1 2 3 4]
y_train[:5] = [3. 3. 5. 3. 5.]


In [8]:
# PMF core functions
# Model:
# r_hat_ui = mu + b_u + b_i + p_u^T q_i

@njit
def predict_one(u, i, global_mean, bu, bi, P, Q):
    pred = global_mean + bu[u] + bi[i]
    
    dot = 0.0
    for f in range(P.shape[1]):
        dot += P[u, f] * Q[i, f]
    pred += dot

    # clip to original 1-5 scale
    if pred < 1.0:
        pred = 1.0
    elif pred > 5.0:
        pred = 5.0

    return pred


@njit
def rmse_score(u_idx, i_idx, y, global_mean, bu, bi, P, Q):
    se = 0.0
    n = len(y)

    for t in range(n):
        pred = predict_one(u_idx[t], i_idx[t], global_mean, bu, bi, P, Q)
        err = y[t] - pred
        se += err * err

    return np.sqrt(se / n)


@njit
def train_pmf(
    u_train, i_train, y_train,
    u_val, i_val, y_val,
    n_users, n_items,
    k, epochs, lr, reg, seed
):
    np.random.seed(seed)

    global_mean = np.mean(y_train)

    P = 0.1 * np.random.randn(n_users, k)
    Q = 0.1 * np.random.randn(n_items, k)
    bu = np.zeros(n_users)
    bi = np.zeros(n_items)

    n = len(y_train)
    history = np.zeros((epochs, 3))   # epoch, train_rmse, val_rmse
    order = np.arange(n)

    for ep in range(epochs):
        np.random.shuffle(order)

        for s in range(n):
            idx = order[s]
            u = u_train[idx]
            i = i_train[idx]
            r = y_train[idx]

            pred = global_mean + bu[u] + bi[i]
            dot = 0.0
            for f in range(k):
                dot += P[u, f] * Q[i, f]
            pred += dot

            # no clipping during gradient step
            err = r - pred

            bu[u] += lr * (err - reg * bu[u])
            bi[i] += lr * (err - reg * bi[i])

            for f in range(k):
                pu = P[u, f]
                qi = Q[i, f]

                P[u, f] += lr * (err * qi - reg * pu)
                Q[i, f] += lr * (err * pu - reg * qi)

        train_rmse = rmse_score(u_train, i_train, y_train, global_mean, bu, bi, P, Q)
        val_rmse = rmse_score(u_val, i_val, y_val, global_mean, bu, bi, P, Q)

        history[ep, 0] = ep + 1
        history[ep, 1] = train_rmse
        history[ep, 2] = val_rmse

    return global_mean, bu, bi, P, Q, history

In [9]:
# Python wrapper
def fit_model(k=20, epochs=10, lr=0.01, reg=0.02, seed=42):
    global_mean, bu, bi, P, Q, history = train_pmf(
        u_train, i_train, y_train,
        u_val, i_val, y_val,
        n_users, n_items,
        k=k, epochs=epochs, lr=lr, reg=reg, seed=seed
    )

    test_rmse = rmse_score(u_test, i_test, y_test, global_mean, bu, bi, P, Q)

    user_counts = np.zeros(n_users, dtype=np.int64)
    item_counts = np.zeros(n_items, dtype=np.int64)

    for u in u_train:
        user_counts[u] += 1
    for i in i_train:
        item_counts[i] += 1

    user_seen = train_df.groupby("u_idx")["i_idx"].apply(set).to_dict()

    model = {
        "global_mean": global_mean,
        "bu": bu,
        "bi": bi,
        "P": P,
        "Q": Q,
        "history": history,
        "test_rmse": test_rmse,
        "user_counts": user_counts,
        "item_counts": item_counts,
        "user_seen": user_seen,
        "k": k,
        "epochs": epochs,
        "lr": lr,
        "reg": reg,
        "seed": seed
    }
    return model

In [10]:
# Prediction + uncertainty
# Bayesian-style approximation
def predict_mean(model, raw_user_id, raw_movie_id):
    if raw_user_id not in user2idx or raw_movie_id not in movie2idx:
        return None

    u = user2idx[raw_user_id]
    i = movie2idx[raw_movie_id]

    pred = (
        model["global_mean"]
        + model["bu"][u]
        + model["bi"][i]
        + np.dot(model["P"][u], model["Q"][i])
    )
    pred = float(np.clip(pred, 1.0, 5.0))
    return pred


def predict_with_uncertainty(model, raw_user_id, raw_movie_id, n_samples=300, temp=0.60):
    pred = predict_mean(model, raw_user_id, raw_movie_id)
    if pred is None:
        return None

    u = user2idx[raw_user_id]
    i = movie2idx[raw_movie_id]

    user_count = model["user_counts"][u]
    item_count = model["item_counts"][i]

    # approximate posterior uncertainty:
    # fewer ratings => higher uncertainty
    var_u = 1.0 / (1.0 + user_count)
    var_i = 1.0 / (1.0 + item_count)
    scale = 0.20 + 0.45 * np.sqrt(var_u + var_i)

    draws = np.random.normal(loc=pred, scale=scale, size=n_samples)
    draws = np.clip(draws, 1.0, 5.0)

    # convert continuous draws into discrete 1-5 probabilities
    stars = np.arange(1, 6)
    weights = np.exp(-0.5 * ((stars[None, :] - draws[:, None]) / temp) ** 2)
    weights = weights / weights.sum(axis=1, keepdims=True)
    rating_probs = weights.mean(axis=0)

    return {
        "mean_pred": float(draws.mean()),
        "low": float(np.quantile(draws, 0.025)),
        "high": float(np.quantile(draws, 0.975)),
        "rating_probs": rating_probs,
        "draws": draws
    }


def recommend_top_n(model, raw_user_id, top_n=10):
    if raw_user_id not in user2idx:
        return None

    u = user2idx[raw_user_id]
    seen = model["user_seen"].get(u, set())

    scores = model["global_mean"] + model["bu"][u] + model["bi"] + model["Q"] @ model["P"][u]
    scores = np.clip(scores, 1.0, 5.0)

    candidates = []
    for i_idx in range(n_items):
        if i_idx not in seen:
            candidates.append((idx2movie[i_idx], float(scores[i_idx])))

    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)[:top_n]
    return pd.DataFrame(candidates, columns=["movie_id", "predicted_rating"])

In [11]:
# Train default model once
model_cache = {"model": None}

default_model = fit_model(
    k=20,
    epochs=10,
    lr=0.01,
    reg=0.02,
    seed=42
)

model_cache["model"] = default_model

print("Default model ready.")
print("Final validation RMSE:", default_model["history"][-1, 2])
print("Test RMSE:", default_model["test_rmse"])

Default model ready.
Final validation RMSE: 0.8704081885785347
Test RMSE: 0.8653530082813704


In [12]:
# Build dashboard
app = JupyterDash(__name__)

default_user = int(train_df.iloc[0]["user_id"])
default_movie = int(train_df.iloc[0]["movie_id"])

app.layout = html.Div([
    html.H1("Bayesian PMF Dashboard"),
    html.P("Core model uses only user_id and movie_id, following the shared project rules."),

    html.Div([
        html.Div([
            html.H3("Training Controls"),

            html.Label("Latent dimension K"),
            dcc.Slider(
                id="k-slider",
                min=4, max=50, step=2, value=20,
                marks={4:"4", 10:"10", 20:"20", 30:"30", 40:"40", 50:"50"}
            ),

            html.Br(),
            html.Label("Epochs"),
            dcc.Slider(
                id="epochs-slider",
                min=3, max=20, step=1, value=10
            ),

            html.Br(),
            html.Label("Learning rate"),
            dcc.Slider(
                id="lr-slider",
                min=0.002, max=0.03, step=0.002, value=0.01
            ),

            html.Br(),
            html.Label("Regularization"),
            dcc.Slider(
                id="reg-slider",
                min=0.005, max=0.08, step=0.005, value=0.02
            ),

            html.Br(),
            html.Button("Train / Refresh Model", id="train-button", n_clicks=0),

            html.Br(), html.Br(),
            html.Div(
                id="train-status",
                style={"whiteSpace": "pre-wrap", "fontWeight": "bold"}
            )
        ], style={
            "width": "30%",
            "display": "inline-block",
            "verticalAlign": "top",
            "border": "1px solid #ccc",
            "padding": "15px",
            "borderRadius": "10px",
            "marginRight": "1%"
        }),

        html.Div([
            dcc.Graph(id="rmse-graph"),
            dcc.Graph(id="latent-summary-graph")
        ], style={
            "width": "68%",
            "display": "inline-block",
            "verticalAlign": "top",
            "border": "1px solid #ccc",
            "padding": "15px",
            "borderRadius": "10px"
        })
    ]),

    html.Br(),

    html.Div([
        html.Div([
            html.H3("Prediction Panel"),

            html.Label("User ID"),
            dcc.Input(
                id="user-input",
                type="number",
                value=default_user,
                style={"width": "100%"}
            ),

            html.Br(), html.Br(),
            html.Label("Movie ID"),
            dcc.Input(
                id="movie-input",
                type="number",
                value=default_movie,
                style={"width": "100%"}
            ),

            html.Br(), html.Br(),
            html.Label("Posterior sample count"),
            dcc.Slider(
                id="posterior-slider",
                min=50, max=500, step=50, value=300
            ),

            html.Br(),
            html.Div(
                id="prediction-box",
                style={
                    "whiteSpace": "pre-wrap",
                    "fontSize": "16px",
                    "border": "1px solid #ccc",
                    "padding": "12px",
                    "borderRadius": "10px"
                }
            )
        ], style={
            "width": "30%",
            "display": "inline-block",
            "verticalAlign": "top",
            "border": "1px solid #ccc",
            "padding": "15px",
            "borderRadius": "10px",
            "marginRight": "1%"
        }),

        html.Div([
            dcc.Graph(id="rating-prob-graph"),
            html.H3("Top Recommendations for User"),
            dash_table.DataTable(
                id="rec-table",
                columns=[
                    {"name": "movie_id", "id": "movie_id"},
                    {"name": "predicted_rating", "id": "predicted_rating"},
                ],
                page_size=10,
                style_table={"overflowX": "auto"}
            )
        ], style={
            "width": "68%",
            "display": "inline-block",
            "verticalAlign": "top",
            "border": "1px solid #ccc",
            "padding": "15px",
            "borderRadius": "10px"
        })
    ])
], style={"fontFamily": "Arial", "margin": "20px"})

/Users/tina2/opt/anaconda3/lib/python3.13/site-packages/dash/dash.py:644: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



In [13]:
# Callbacks
@app.callback(
    Output("train-status", "children"),
    Output("rmse-graph", "figure"),
    Output("latent-summary-graph", "figure"),
    Input("train-button", "n_clicks"),
    State("k-slider", "value"),
    State("epochs-slider", "value"),
    State("lr-slider", "value"),
    State("reg-slider", "value")
)
def retrain_model(n_clicks, k, epochs, lr, reg):
    model = fit_model(
        k=int(k),
        epochs=int(epochs),
        lr=float(lr),
        reg=float(reg),
        seed=42
    )
    model_cache["model"] = model

    hist = pd.DataFrame(model["history"], columns=["epoch", "train_rmse", "val_rmse"])

    fig_rmse = go.Figure()
    fig_rmse.add_trace(
        go.Scatter(
            x=hist["epoch"],
            y=hist["train_rmse"],
            mode="lines+markers",
            name="Train RMSE"
        )
    )
    fig_rmse.add_trace(
        go.Scatter(
            x=hist["epoch"],
            y=hist["val_rmse"],
            mode="lines+markers",
            name="Validation RMSE"
        )
    )
    fig_rmse.update_layout(
        title="Train / Validation RMSE",
        xaxis_title="Epoch",
        yaxis_title="RMSE",
        height=400
    )

    latent_norms_user = np.linalg.norm(model["P"], axis=1)
    latent_norms_item = np.linalg.norm(model["Q"], axis=1)

    fig_latent = go.Figure()
    fig_latent.add_trace(
        go.Histogram(
            x=latent_norms_user,
            name="User latent norm",
            opacity=0.7
        )
    )
    fig_latent.add_trace(
        go.Histogram(
            x=latent_norms_item,
            name="Movie latent norm",
            opacity=0.7
        )
    )
    fig_latent.update_layout(
        barmode="overlay",
        title="Latent Factor Norm Distribution",
        height=400
    )

    status = (
        f"Model trained.\n"
        f"K = {model['k']}\n"
        f"Epochs = {model['epochs']}\n"
        f"Learning rate = {model['lr']}\n"
        f"Regularization = {model['reg']}\n"
        f"Seed = {model['seed']}\n"
        f"Final validation RMSE = {hist['val_rmse'].iloc[-1]:.4f}\n"
        f"Test RMSE = {model['test_rmse']:.4f}"
    )

    return status, fig_rmse, fig_latent


@app.callback(
    Output("prediction-box", "children"),
    Output("rating-prob-graph", "figure"),
    Output("rec-table", "data"),
    Input("user-input", "value"),
    Input("movie-input", "value"),
    Input("posterior-slider", "value")
)
def update_prediction(raw_user_id, raw_movie_id, posterior_samples):
    model = model_cache.get("model", None)

    if model is None:
        fig_empty = go.Figure()
        fig_empty.update_layout(title="No model trained yet")
        return "No model trained yet.", fig_empty, []

    if raw_user_id is None or raw_movie_id is None:
        fig_empty = go.Figure()
        fig_empty.update_layout(title="Missing input")
        return "Please enter both user_id and movie_id.", fig_empty, []

    raw_user_id = int(raw_user_id)
    raw_movie_id = int(raw_movie_id)

    pred_info = predict_with_uncertainty(
        model,
        raw_user_id,
        raw_movie_id,
        n_samples=int(posterior_samples)
    )

    if pred_info is None:
        fig_empty = go.Figure()
        fig_empty.update_layout(title="Unknown user_id or movie_id")
        return "user_id or movie_id not found.", fig_empty, []

    prob_df = pd.DataFrame({
        "rating": [1, 2, 3, 4, 5],
        "probability": pred_info["rating_probs"]
    })

    fig_prob = px.bar(
        prob_df,
        x="rating",
        y="probability",
        title="Probability of Each Discrete Rating"
    )
    fig_prob.update_layout(height=400)

    rec_df = recommend_top_n(model, raw_user_id, top_n=10)
    if rec_df is None:
        rec_data = []
    else:
        rec_df["predicted_rating"] = rec_df["predicted_rating"].round(4)
        rec_data = rec_df.to_dict("records")

    text = (
        f"Predicted mean rating: {pred_info['mean_pred']:.4f}\n"
        f"95% interval: [{pred_info['low']:.4f}, {pred_info['high']:.4f}]\n"
        f"Most likely rating: {int(np.argmax(pred_info['rating_probs']) + 1)}\n"
        f"All predictions are clipped to [1,5].\n"
        f"Metric used: RMSE on original 1-5 scale."
    )

    return text, fig_prob, rec_data

In [14]:
app.run(jupyter_mode="inline", debug=False, port=8051)